# ANOVA: Plant Growth

Each block is editable. Contrasts use the printed group order.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT))
import pandas as pd
from anova_toolbox import ANOVAToolbox

# Dataset choice
# Keep True for the included default dataset.
# Set to False to use your own dataset, then enter the file name WITH extension.
# Put custom files either in the project folder or in the data/ folder.
use_default_dataset = True
custom_file_name = "your_anova_data.csv"  # examples: my_data.csv, my_data.xlsx, my_data.json

def choose_data_file(default_file_name, custom_file_name, use_default_dataset):
    if use_default_dataset:
        return ROOT / 'data' / default_file_name
    if not custom_file_name or custom_file_name == "your_anova_data.csv":
        raise ValueError("Set custom_file_name to your file name with extension, for example 'my_data.csv'.")
    custom_path = Path(custom_file_name)
    if custom_path.suffix.lower() not in {'.csv', '.xlsx', '.xls', '.json'}:
        raise ValueError("Custom file must include a supported extension: .csv, .xlsx, .xls, or .json")
    candidates = [custom_path, ROOT / custom_path, ROOT / 'data' / custom_path.name]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {custom_file_name}. Put it in the project folder or data/ folder.")

def validate_for_anova(df, group_col, value_col):
    missing = [c for c in [group_col, value_col] if c not in df.columns]
    if missing:
        raise ValueError(f"Dataset is missing required columns for ANOVA: {missing}. Available columns: {list(df.columns)}")
    numeric = pd.to_numeric(df[value_col], errors='coerce')
    usable = df.assign(_value=numeric).dropna(subset=[group_col, '_value'])
    if usable[group_col].nunique() < 2:
        raise ValueError(f"ANOVA needs at least two groups in '{group_col}'.")
    if usable.groupby(group_col)['_value'].count().min() < 2:
        raise ValueError("Each ANOVA group should have at least two numeric observations.")

# If using a custom dataset, update these column names and the later test blocks to match your file.
group_col = 'group'
value_col = 'weight'

data_path = choose_data_file('plant_growth.csv', custom_file_name, use_default_dataset)
print('Loaded file:', data_path)
tool = ANOVAToolbox(data_path)
df = tool.data
validate_for_anova(df, group_col, value_col)
display(df.head())
print('Shape:', df.shape)
display(df.isna().sum().to_frame('missing'))
display(tool.summarize_data(group_col, value_col))
print('Group order for contrasts:', list(tool.group_summary(group_col, value_col).index))

## Block 1: One-way ANOVA

In [ ]:
group_col = 'group'
value_col = 'weight'
alpha = 0.05

result = tool.one_way_anova(group_col, value_col, alpha)
print('H0: all treatment means are equal')
print('H1: at least one treatment mean differs')
display(tool.anova_table(result))
print('Decision:', result['decision'])
tool.plot_f_distribution(result)
tool.plot_group_boxplot(group_col, value_col)
tool.plot_group_means_ci(group_col, value_col);

## Block 2: Contrast test
For group order `ctrl`, `trt1`, `trt2`, the contrast `[-1, 0, 1]` tests trt2 minus control.

In [ ]:
group_col = 'group'
value_col = 'weight'
contrast_vector = [-1, 0, 1]
null_value = 0
alternative = 'greater'
alpha = 0.05

contrast = tool.contrast_test(group_col, value_col, contrast_vector, null_value, alternative, alpha)
display(__import__('pandas').DataFrame({k: [v] for k, v in contrast.items() if k not in ['groups', 'contrast_vector']}))
print('Groups:', contrast['groups'])
print('Contrast vector:', contrast['contrast_vector'])
tool.plot_t_distribution_for_contrast(contrast);

## Block 3: Linear combination confidence interval

In [ ]:
weights = [0, -1, 1]
confidence = 0.95

ci = tool.linear_combination_ci('group', 'weight', weights, confidence)
display(__import__('pandas').DataFrame({k: [v] for k, v in ci.items()}))

## Block 4: Pairwise comparisons and residual diagnostics

In [ ]:
display(tool.bonferroni_pairwise('group', 'weight', alpha=0.05))
result = tool.one_way_anova('group', 'weight')
tool.plot_residual_diagnostics(result);